# Ultra Low Complexity Noise Suppression (ULCNet - Simplified)

This notebook implements an end-to-end simplified version of the ULCNet model
based on the paper "Ultra Low Complexity Deep Learning Based Noise Suppression".

Features:

- STFT + Power Law Compression
- Two-stage Network (CRN + CNN)
- Training + Inference
- Audio Reconstruction


In [ ]:

# Install dependencies (run once)
# !pip install torch torchaudio librosa soundfile tqdm


In [2]:

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import librosa
import numpy as np
import soundfile as sf
from tqdm import tqdm
import os


## STFT and Power-Law Compression


In [3]:
# ---- Collate function for DataLoader --------------------------------------
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    """
    batch: list of tuples (mix_tensor, clean_tensor) with variable lengths
    Returns:
      mix_padded: [B, Lmax]
      clean_padded: [B, Lmax]
      lengths: tensor([L1, L2, ...])
    """
    mixes = [b[0].float() for b in batch]
    cleans = [b[1].float() for b in batch]
    lengths = torch.tensor([m.shape[0] for m in mixes], dtype=torch.long)

    mix_p = pad_sequence(mixes, batch_first=True)   # [B, Lmax]
    clean_p = pad_sequence(cleans, batch_first=True) # [B, Lmax]

    return mix_p, clean_p, lengths


In [4]:

def power_compress(x, alpha=0.3):
    return torch.sign(x) * torch.abs(x) ** alpha

def power_decompress(x, alpha=0.3):
    return torch.sign(x) * torch.abs(x) ** (1/alpha)


# ---- Batched STFT / ISTFT helpers ----------------------------------------
# Updated to support batched signals (B, L) or single (L,)
def compute_stft(wav, n_fft=512, hop=256, win_length=512):
    # If mono 1D -> make [1, L]; if already batched (B,L) pass through.
    if wav.dim() == 1:
        wav = wav.unsqueeze(0)
    # returns complex tensor shape [B, freq_bins, frames]
    return torch.stft(wav, n_fft=n_fft, hop_length=hop, win_length=win_length,
                      return_complex=True, center=True)

def compute_istft(spec, n_fft=512, hop=256, win_length=512, length=None):
    """
    spec: complex tensor [B, freq_bins, frames] OR [freq_bins, frames] for single
    Returns waveform [B, L] or [L] if single
    """
    # torch.istft supports batched complex input
    return torch.istft(spec, n_fft=n_fft, hop_length=hop, win_length=win_length,
                       length=length, center=True)



## Stage 1: CRN (Magnitude Mask)


In [5]:

class CRNStage(nn.Module):
    def __init__(self, freq_bins=257):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, (1,3), padding=(0,1)),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.Conv2d(32, 64, (1,3), padding=(0,1)),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(64, 128, (1,3), padding=(0,1)),
            nn.BatchNorm2d(128),
            nn.ReLU()
        )

        self.gru = nn.GRU(
            input_size=128*freq_bins,
            hidden_size=256,
            batch_first=True,
            bidirectional=True
        )

        self.fc = nn.Sequential(
            nn.Linear(512, freq_bins),
            nn.Sigmoid()
        )


    def forward(self, x):
        # x: [B,1,T,F]
        B,C,T,F = x.shape

        x = self.conv(x)

        x = x.permute(0,2,1,3)
        x = x.reshape(B, T, -1)

        x,_ = self.gru(x)

        mask = self.fc(x)

        return mask


## Stage 2: CNN (Complex Mask)


In [6]:

class CNNStage(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(2, 32, (1,3), padding=(0,1)),
            nn.ReLU(),

            nn.Conv2d(32, 32, (1,3), padding=(0,1)),
            nn.ReLU(),

            nn.Conv2d(32, 2, 1)
        )


    def forward(self, x):
        return self.net(x)


## Full ULCNet Model


In [7]:

class ULCNet(nn.Module):
    def __init__(self, freq_bins=257, alpha=0.3):
        super().__init__()

        self.alpha = alpha

        self.stage1 = CRNStage(freq_bins)
        self.stage2 = CNNStage()


    def forward(self, mag, phase):

        # Stage 1
        mag = mag.unsqueeze(1)
        mask = self.stage1(mag)

        # Intermediate features
        yr = mask * torch.cos(phase)
        yi = mask * torch.sin(phase)

        feat = torch.stack([yr, yi], dim=1)

        # Stage 2
        cmask = self.stage2(feat)

        mr = cmask[:,0]
        mi = cmask[:,1]

        return mr, mi


## Dataset Loader (DNS-Style Simulation)


In [8]:

class SimpleDataset(torch.utils.data.Dataset):

    def __init__(self, clean_files, noise_files, sr=16000):
        self.clean = clean_files
        self.noise = noise_files
        self.sr = sr


    def __len__(self):
        return len(self.clean)


    def __getitem__(self, idx):

        c,_ = librosa.load(self.clean[idx], sr=self.sr)
        n,_ = librosa.load(
            np.random.choice(self.noise),
            sr=self.sr
        )

        L = min(len(c), len(n))
        c = c[:L]
        n = n[:L]

        snr = np.random.uniform(-5,15)
        n = n * 10**(-snr/20)

        mix = c + n

        return torch.tensor(mix), torch.tensor(c)


## Training Loop


In [9]:
# ---- Updated training loop (masks out padded samples) ---------------------
def train(model, loader, optimizer, device, n_fft=512, hop=256, alpha=0.3):
    model.train()
    total_loss = 0.0
    total_batches = 0

    for mix, clean, lengths in tqdm(loader):
        mix = mix.to(device)       # [B, Lmax]
        clean = clean.to(device)   # [B, Lmax]
        lengths = lengths.to(device)

        # 1) STFT (batched)
        X = compute_stft(mix, n_fft=n_fft, hop=hop)    # [B, F, T] complex
        # Clean STFT available if needed, but we'll compute time-domain target (clean)

        # 2) Power-law compress real/imag
        Xr = power_compress(X.real, alpha=alpha)
        Xi = power_compress(X.imag, alpha=alpha)

        # 3) Magnitude and phase - align shapes to [B, T, F] for model input
        mag = torch.sqrt(Xr**2 + Xi**2)   # [B, F, T]
        phase = torch.atan2(Xi, Xr)       # [B, F, T]

        mag = mag.permute(0,2,1)   # -> [B, T, F]
        phase = phase.permute(0,2,1) # -> [B, T, F]

        # 4) Forward through model (model expects mag [B,1,T,F] in the original notebook)
        mr, mi = model(mag, phase)  # keep same contract: mr,mi -> [B, T, F]

        # Bring masks back to [B, F, T] to multiply with Xr/Xi
        mr = mr.permute(0,2,1)  # [B, F, T]
        mi = mi.permute(0,2,1)  # [B, F, T]

        # 5) Compose estimated real/imag in compressed domain
        est_r_c = mr * Xr - mi * Xi   # compressed-domain estimated real [B,F,T]
        est_i_c = mr * Xi + mi * Xr

        # 6) Power-decompress to original scale
        est_r = power_decompress(est_r_c, alpha=alpha)
        est_i = power_decompress(est_i_c, alpha=alpha)

        # 7) Reconstruct time-domain signals (batched)
        spec_est = torch.complex(est_r, est_i)  # [B, F, T]
        # torch.istft expects (B, F, T) complex -> returns [B, Lrecon]
        # We need to supply length parameter = original padded length (Lmax) to ensure consistent output length
        Lmax = mix.shape[1]
        out_wav = compute_istft(spec_est, n_fft=n_fft, hop=hop, win_length=n_fft, length=Lmax)  # [B, Lmax]

        # 8) Compute MSE loss only over original lengths (mask out padded portion)
        # We'll compute sum MSE and divide by total number of valid samples
        batch_loss = 0.0
        total_valid = 0
        for i, l in enumerate(lengths):
            li = int(l.item())
            if li == 0:
                continue
            # use reduction='sum' to weight by length
            batch_loss += F.mse_loss(out_wav[i,:li], clean[i,:li], reduction='sum')
            total_valid += li

        if total_valid == 0:
            # fallback (shouldn't happen)
            loss = torch.tensor(0.0, device=device)
        else:
            loss = batch_loss / total_valid

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_batches += 1

    return total_loss / max(1, total_batches)


## Inference / Enhancement


In [10]:
# ---- Inference: accept (wav, original_length) and crop output ------------
def enhance(model, wav, device, n_fft=512, hop=256, alpha=0.3):
    """
    wav: 1D torch tensor or numpy array (variable length)
    returns enhanced clipped to original length
    """
    model.eval()
    if isinstance(wav, np.ndarray):
        wav = torch.tensor(wav, dtype=torch.float32)

    orig_len = wav.shape[0]
    wav_b = wav.unsqueeze(0).to(device)  # [1, L]

    with torch.no_grad():
        X = compute_stft(wav_b, n_fft=n_fft, hop=hop)  # [1, F, T] complex

        Xr = power_compress(X.real, alpha=alpha)
        Xi = power_compress(X.imag, alpha=alpha)

        mag = torch.sqrt(Xr**2 + Xi**2).permute(0,2,1)  # [B=1, T, F]
        phase = torch.atan2(Xi, Xr).permute(0,2,1)      # [1, T, F]

        mr, mi = model(mag, phase)  # [1, T, F] each
        mr = mr.permute(0,2,1)  # [1, F, T]
        mi = mi.permute(0,2,1)

        est_r_c = mr * Xr - mi * Xi
        est_i_c = mr * Xi + mi * Xr

        est_r = power_decompress(est_r_c, alpha=alpha)
        est_i = power_decompress(est_i_c, alpha=alpha)

        spec = torch.complex(est_r, est_i)

        # ISTFT with length=orig_len ensures we get same length as original
        out = compute_istft(spec, n_fft=n_fft, hop=hop, win_length=n_fft, length=orig_len)  # [1, orig_len]
        out = out.squeeze(0).cpu().numpy()

    return out  # already cropped to orig_len


## Example Usage


In [11]:
# training params
clean_dir = "clean_trainset_wav"
noisy_dir = "noisy_trainset_wav"

# testing params
test_file = "app2_data/noisy/p232_003.wav"
output_location = "enhanced_output.wav"

In [12]:

# Example (requires your own wav files)

device = "cuda" if torch.cuda.is_available() else "cpu"

model = ULCNet().to(device)
opt = torch.optim.Adam(model.parameters(), lr=4e-4)

# Prepare dataset
clean_files = []
noise_files = []

# clean files append
for filename in os.listdir(clean_dir):
    if filename.endswith(".wav"):
        clean_files.append(os.path.join(clean_dir, filename))
# noise files append
for filename in os.listdir(noisy_dir):
    if filename.endswith(".wav"):
        noise_files.append(os.path.join(noisy_dir, filename))
  
# Uncomment for quick testing
# clean_files = clean_files[:5]
# noise_files = noise_files[:5]
    
# ---- DataLoader usage: pass collate_fn -----------------------------------
# Example:
dataset = SimpleDataset(clean_files, noise_files, sr=16000)  # unchanged dataset OK
loader = torch.utils.data.DataLoader(
    dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0   # IMPORTANT on Windows
)



for epoch in range(10):
    loss = train(model, loader, opt, device)
    print("Epoch", epoch, loss)
    torch.save(model.state_dict(), f"ulcnet_model_{epoch}.pth")

# Enhance
wav,_ = librosa.load(test_file, sr=16000)
out = enhance(model, torch.tensor(wav), device)
sf.write(output_location, out, 16000)


  0%|          | 0/2893 [00:00<?, ?it/s]c:\Users\Aryan Gupta\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\functional.py:709: UserWarning: A window was not provided. A rectangular window will be applied,which is known to cause spectral leakage. Other windows such as torch.hann_window or torch.hamming_window can are recommended to reduce spectral leakage.To suppress this warning and use a rectangular window, explicitly set `window=torch.ones(n_fft, device=<device>)`. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\SpectralOps.cpp:842.)
  return _VF.stft(  # type: ignore[attr-defined]
C:\Users\Aryan Gupta\AppData\Local\Temp\ipykernel_31440\3557808025.py:24: UserWarning: A window was not provided. A rectangular window will be applied.Please provide the same window used by stft to make the inversion lossless.To suppress this warning and use a rectangular window, explicitly set `window=torch.ones(n_fft, device=<device>)`. (Trig

Epoch 0 0.001793917549834894


100%|██████████| 2893/2893 [12:11:45<00:00, 15.18s/it]        


Epoch 1 0.0016792139867676382


 12%|█▏        | 354/2893 [19:32<2:20:12,  3.31s/it]   


KeyboardInterrupt: 

## Save Model


In [13]:
# add a way to save and load the model
# torch.save(model.state_dict(), "ulcnet_model.pth")
# Load the model (example)
model.load_state_dict(torch.load("ulcnet_model_1.pth"))

<All keys matched successfully>

In [14]:
# Enhance
wav,_ = librosa.load(test_file, sr=16000)
out = enhance(model, torch.tensor(wav), device)
sf.write(output_location, out, 16000)

## Notes

This is a faithful but simplified implementation for learning and experimentation.

For production, full subband GRUs and channelwise reorientation should be added.
